# Social Media Network Analysis

This notebook examines a social network by constructing a graph from interaction data and analyzing its structure through metrics like centrality, clustering, and community patterns. It highlights key influencers, visualizes how users connect, and interprets how information might spread across the network. The analysis concludes with a discussion translating these findings into practical insights for shaping an effective social media strategy.

---
## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Synthetic Interaction Data Generation](#2-synthetic-interaction-data-generation)
3. [Graph Construction](#3-graph-construction)
4. [Basic Network Statistics](#4-basic-network-statistics)
5. [Centrality Analysis](#5-centrality-analysis)
6. [Clustering Coefficients](#6-clustering-coefficients)
7. [Community Detection](#7-community-detection)
8. [Key Influencer Identification](#8-key-influencer-identification)
9. [Information Spread Simulation](#9-information-spread-simulation)
10. [Network Visualizations](#10-network-visualizations)
11. [Social Media Strategy Discussion](#11-social-media-strategy-discussion)

## 1. Setup & Imports

In [ ]:
# Install required libraries if not already present
import subprocess, sys

required = [
    "networkx",
    "matplotlib",
    "pandas",
    "numpy",
    "seaborn",
    "community",   # python-louvain
]

for pkg in required:
    try:
        __import__(pkg if pkg != "community" else "community")
    except ImportError:
        install_name = "python-louvain" if pkg == "community" else pkg
        subprocess.check_call([sys.executable, "-m", "pip", "install", install_name, "-q"])

In [ ]:
import random
import itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Attempt to import python-louvain; fall back to NetworkX greedy modularity
try:
    import community as community_louvain
    LOUVAIN_AVAILABLE = True
except ImportError:
    LOUVAIN_AVAILABLE = False
    print("python-louvain not available; using NetworkX greedy modularity for community detection.")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 6)})
print("NetworkX version:", nx.__version__)

---
## 2. Synthetic Interaction Data Generation

We simulate a realistic social network with **150 users** distributed across five persona groups (Power Users, Casual Connectors, Niche Commenters, Lurkers, and Bridge Nodes). Interactions (likes, comments, shares, replies) are sampled with probabilities reflecting each persona's activity level.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
N_USERS = 150

PERSONA_CONFIG = {
    "Power User":         {"count": 10,  "activity": 1.0,  "color": "#e74c3c"},
    "Casual Connector":   {"count": 40,  "activity": 0.55, "color": "#3498db"},
    "Niche Commenter":    {"count": 35,  "activity": 0.45, "color": "#2ecc71"},
    "Lurker":             {"count": 50,  "activity": 0.15, "color": "#95a5a6"},
    "Bridge Node":        {"count": 15,  "activity": 0.75, "color": "#f39c12"},
}

INTERACTION_TYPES = ["like", "comment", "share", "reply"]
INTERACTION_WEIGHTS = [0.50, 0.25, 0.15, 0.10]

# ── Build user list ────────────────────────────────────────────────────────────
users = []
user_persona = {}
uid = 0
for persona, cfg in PERSONA_CONFIG.items():
    for _ in range(cfg["count"]):
        users.append(f"user_{uid:03d}")
        user_persona[f"user_{uid:03d}"] = persona
        uid += 1

print(f"Total users: {len(users)}")
print("Persona distribution:")
for p, cfg in PERSONA_CONFIG.items():
    print(f"  {p}: {cfg['count']}")

In [ ]:
def generate_interactions(users, user_persona, persona_config, seed=SEED):
    """Generate a list of (source, target, interaction_type, weight) tuples."""
    rng = random.Random(seed)
    interactions = []

    for i, src in enumerate(users):
        src_persona = user_persona[src]
        src_activity = persona_config[src_persona]["activity"]

        # Determine how many interactions this user initiates
        n_interactions = rng.randint(
            int(2 * src_activity),
            int(20 * src_activity) + 1,
        )

        # Prefer interactions with "nearby" users + some long-range connections
        candidates = users[:]
        candidates.remove(src)
        # Bias toward first ~30 users to create denser core
        weighted_candidates = candidates[:30] * 3 + candidates

        targets = rng.choices(weighted_candidates, k=n_interactions)
        for tgt in targets:
            if tgt == src:
                continue
            itype = rng.choices(INTERACTION_TYPES, weights=INTERACTION_WEIGHTS, k=1)[0]
            weight = {"like": 1, "comment": 2, "share": 3, "reply": 2}[itype]
            interactions.append((src, tgt, itype, weight))

    return interactions


interactions = generate_interactions(users, user_persona, PERSONA_CONFIG)
df = pd.DataFrame(interactions, columns=["source", "target", "interaction_type", "weight"])
print(f"Total interactions generated: {len(df):,}")
df.head(10)

---
## 3. Graph Construction

We build a **weighted directed graph** where nodes are users and edges represent aggregated interactions.  
Each edge stores the total interaction weight and the interaction count.

In [ ]:
# Aggregate edges: sum weights and count interactions per (source, target) pair
edge_df = (
    df.groupby(["source", "target"])
    .agg(weight=("weight", "sum"), interaction_count=("weight", "count"))
    .reset_index()
)

# Build directed weighted graph
DG = nx.DiGraph()
DG.add_nodes_from(users)
for _, row in edge_df.iterrows():
    DG.add_edge(
        row["source"],
        row["target"],
        weight=row["weight"],
        interaction_count=row["interaction_count"],
    )

# Assign node attributes
nx.set_node_attributes(DG, user_persona, name="persona")

# Also create undirected version for some analyses
G = DG.to_undirected()

print(f"Directed graph  — Nodes: {DG.number_of_nodes():,}  |  Edges: {DG.number_of_edges():,}")
print(f"Undirected graph — Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}")

---
## 4. Basic Network Statistics

Before diving into advanced metrics, we characterize the overall topology of the network.

In [ ]:
# Density
density = nx.density(DG)

# Largest weakly-connected component
wcc = max(nx.weakly_connected_components(DG), key=len)
DG_lcc = DG.subgraph(wcc).copy()
G_lcc = DG_lcc.to_undirected()

# Average clustering (undirected LCC)
avg_clustering = nx.average_clustering(G_lcc)

# Average shortest path length (undirected LCC)
avg_path_len = nx.average_shortest_path_length(G_lcc)

# Degree sequence
in_degrees  = dict(DG.in_degree())
out_degrees = dict(DG.out_degree())

stats = {
    "Nodes": DG.number_of_nodes(),
    "Directed Edges": DG.number_of_edges(),
    "Graph Density": round(density, 5),
    "LCC Size (nodes)": len(wcc),
    "Average Clustering Coefficient": round(avg_clustering, 4),
    "Average Shortest Path Length": round(avg_path_len, 4),
    "Max In-Degree": max(in_degrees.values()),
    "Max Out-Degree": max(out_degrees.values()),
}

stats_df = pd.DataFrame.from_dict(stats, orient="index", columns=["Value"])
print("=== Network Summary Statistics ===")
print(stats_df.to_string())

In [ ]:
# Degree distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

in_deg_vals  = list(in_degrees.values())
out_deg_vals = list(out_degrees.values())

axes[0].hist(in_deg_vals, bins=30, color="#3498db", edgecolor="white", alpha=0.85)
axes[0].set_title("In-Degree Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("In-Degree")
axes[0].set_ylabel("Number of Users")

axes[1].hist(out_deg_vals, bins=30, color="#e74c3c", edgecolor="white", alpha=0.85)
axes[1].set_title("Out-Degree Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Out-Degree")
axes[1].set_ylabel("Number of Users")

plt.suptitle("Social Network Degree Distributions", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("degree_distribution.png", bbox_inches="tight")
plt.show()
print("Figure saved: degree_distribution.png")

---
## 5. Centrality Analysis

Centrality measures reveal **how important** each node is within the network:

| Metric | What it measures |
|---|---|
| **Degree Centrality** | Raw number of connections (in + out) |
| **In-Degree Centrality** | How many users interact *toward* this node (popularity/reach) |
| **Betweenness Centrality** | How often a node lies on shortest paths (broker/bridge role) |
| **Closeness Centrality** | How quickly a node can reach all others (speed of info spread) |
| **Eigenvector Centrality** | Influence weighted by the influence of neighbors (prestige) |

In [ ]:
print("Computing centrality metrics (may take a moment)...")

degree_cent      = nx.degree_centrality(DG)
in_degree_cent   = nx.in_degree_centrality(DG)
out_degree_cent  = nx.out_degree_centrality(DG)
betweenness_cent = nx.betweenness_centrality(DG, weight="weight", normalized=True)
closeness_cent   = nx.closeness_centrality(DG)

# Eigenvector centrality on undirected LCC (directed version can fail on weakly-connected graphs)
try:
    eigenvector_cent = nx.eigenvector_centrality(DG, max_iter=500, weight="weight")
except nx.PowerIterationFailedConvergence:
    eigenvector_cent = nx.eigenvector_centrality_numpy(G_lcc, weight="weight")
    # fill missing nodes with 0
    eigenvector_cent = {n: eigenvector_cent.get(n, 0.0) for n in DG.nodes()}

# Assemble into a DataFrame
centrality_df = pd.DataFrame(
    {
        "degree":      degree_cent,
        "in_degree":   in_degree_cent,
        "out_degree":  out_degree_cent,
        "betweenness": betweenness_cent,
        "closeness":   closeness_cent,
        "eigenvector": eigenvector_cent,
        "persona":     user_persona,
    }
).rename_axis("user")

print("Centrality computation complete.")
centrality_df.sort_values("in_degree", ascending=False).head(10)

In [ ]:
# Correlation heat-map between centrality measures
cent_cols = ["degree", "in_degree", "out_degree", "betweenness", "closeness", "eigenvector"]
corr = centrality_df[cent_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Centrality Metrics Correlation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("centrality_correlation.png", bbox_inches="tight")
plt.show()
print("Figure saved: centrality_correlation.png")

In [ ]:
# Top-10 nodes by each centrality metric
print("=== Top-10 Users by Betweenness Centrality (Brokers) ===")
print(centrality_df[["betweenness", "persona"]].sort_values("betweenness", ascending=False).head(10).to_string())

print("\n=== Top-10 Users by Eigenvector Centrality (Prestige) ===")
print(centrality_df[["eigenvector", "persona"]].sort_values("eigenvector", ascending=False).head(10).to_string())

---
## 6. Clustering Coefficients

The **clustering coefficient** of a node measures the fraction of its neighbors that are also connected to each other — a proxy for how "tight-knit" the local community around a user is.

- A value near **1.0** indicates the user is embedded in a dense clique.
- A value near **0.0** indicates the user connects otherwise unconnected groups.

In [ ]:
clustering = nx.clustering(G)           # local clustering (undirected)
centrality_df["clustering"] = pd.Series(clustering)

print(f"Average clustering coefficient: {np.mean(list(clustering.values())):.4f}")

# Clustering by persona
cluster_by_persona = (
    centrality_df.groupby("persona")["clustering"]
    .agg(["mean", "median", "std"])
    .rename(columns={"mean": "Mean CC", "median": "Median CC", "std": "Std CC"})
    .sort_values("Mean CC", ascending=False)
)
print("\nClustering coefficient by persona:")
print(cluster_by_persona.to_string())

In [ ]:
# Box plot: clustering by persona
fig, ax = plt.subplots(figsize=(12, 5))
persona_order = cluster_by_persona.index.tolist()
palette = {p: PERSONA_CONFIG[p]["color"] for p in persona_order}

sns.boxplot(
    data=centrality_df.reset_index(),
    x="persona",
    y="clustering",
    order=persona_order,
    palette=palette,
    ax=ax,
    linewidth=1.2,
)
ax.set_title("Local Clustering Coefficient by User Persona", fontsize=13, fontweight="bold")
ax.set_xlabel("Persona")
ax.set_ylabel("Clustering Coefficient")
plt.tight_layout()
plt.savefig("clustering_by_persona.png", bbox_inches="tight")
plt.show()
print("Figure saved: clustering_by_persona.png")

---
## 7. Community Detection

Community detection reveals **groups of users** that interact more densely with each other than with the rest of the network.  
We use the **Louvain algorithm** (or NetworkX greedy modularity as a fallback) on the undirected weighted graph.

- **Modularity (Q)** quantifies the quality of the partition; Q > 0.3 is generally considered meaningful.

In [ ]:
if LOUVAIN_AVAILABLE:
    partition = community_louvain.best_partition(G, weight="weight", random_state=SEED)
    modularity = community_louvain.modularity(partition, G, weight="weight")
    method_name = "Louvain"
else:
    # Greedy modularity (NetworkX built-in)
    communities_gen = nx.community.greedy_modularity_communities(G, weight="weight")
    communities_list = list(communities_gen)
    partition = {}
    for cid, community_set in enumerate(communities_list):
        for node in community_set:
            partition[node] = cid
    modularity = nx.community.modularity(
        G,
        communities_list,
        weight="weight",
    )
    method_name = "Greedy Modularity"

centrality_df["community"] = pd.Series(partition)
nx.set_node_attributes(DG, partition, name="community")

community_counts = Counter(partition.values())
n_communities = len(community_counts)

print(f"Method: {method_name}")
print(f"Number of communities detected: {n_communities}")
print(f"Modularity (Q): {modularity:.4f}")
print("\nCommunity sizes:")
for cid, size in sorted(community_counts.items(), key=lambda x: -x[1]):
    print(f"  Community {cid}: {size} users")

In [ ]:
# Community composition: which personas are in each community?
community_persona = (
    centrality_df.groupby(["community", "persona"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(12, 6))
community_persona.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[PERSONA_CONFIG[p]["color"] for p in community_persona.columns],
    edgecolor="white",
    linewidth=0.5,
)
ax.set_title("Community Composition by User Persona", fontsize=13, fontweight="bold")
ax.set_xlabel("Community ID")
ax.set_ylabel("User Count")
ax.legend(title="Persona", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("community_composition.png", bbox_inches="tight")
plt.show()
print("Figure saved: community_composition.png")

---
## 8. Key Influencer Identification

We define a composite **Influencer Score** that combines multiple centrality signals:

$$\text{Influencer Score} = 0.35 \cdot \hat{C}_{in} + 0.30 \cdot \hat{C}_{eigen} + 0.20 \cdot \hat{C}_{between} + 0.15 \cdot \hat{C}_{close}$$

where each $\hat{C}$ is the min-max normalized value of the corresponding centrality metric.

This scoring reflects that **reach** (in-degree) and **prestige** (eigenvector) are the primary drivers of influence, while **brokerage** (betweenness) and **speed** (closeness) add secondary value.

In [ ]:
def minmax_norm(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0.0
    return (series - mn) / (mx - mn)


centrality_df["influencer_score"] = (
    0.35 * minmax_norm(centrality_df["in_degree"])
    + 0.30 * minmax_norm(centrality_df["eigenvector"])
    + 0.20 * minmax_norm(centrality_df["betweenness"])
    + 0.15 * minmax_norm(centrality_df["closeness"])
)

top_influencers = centrality_df.sort_values("influencer_score", ascending=False).head(15)
print("=== Top-15 Influencers ===")
print(
    top_influencers[
        ["persona", "community", "in_degree", "eigenvector", "betweenness", "closeness", "influencer_score"]
    ]
    .round(4)
    .to_string()
)

In [ ]:
# Bar chart of top-15 influencers
fig, ax = plt.subplots(figsize=(12, 6))
colors = [PERSONA_CONFIG[p]["color"] for p in top_influencers["persona"]]
bars = ax.barh(
    top_influencers.index,
    top_influencers["influencer_score"],
    color=colors,
    edgecolor="white",
)
ax.invert_yaxis()
ax.set_title("Top-15 Influencers by Composite Influencer Score", fontsize=13, fontweight="bold")
ax.set_xlabel("Influencer Score")
ax.set_ylabel("User")

# Custom legend
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=cfg["color"], label=p)
    for p, cfg in PERSONA_CONFIG.items()
]
ax.legend(handles=legend_handles, title="Persona", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("top_influencers.png", bbox_inches="tight")
plt.show()
print("Figure saved: top_influencers.png")

---
## 9. Information Spread Simulation

We simulate the **SIR (Susceptible–Infected–Recovered)** model on the directed network to model how a piece of content (e.g., a viral post) spreads from a seed node.

- **Susceptible**: has not yet seen the content
- **Infected**: actively sharing the content
- **Recovered**: saw the content but is no longer sharing

We compare spread starting from:
1. The **top influencer** (highest composite score)
2. A **random average user** (median influencer score)

In [ ]:
def sir_simulation(
    graph: nx.DiGraph,
    seed_node: str,
    beta: float = 0.15,   # infection probability per edge per step
    gamma: float = 0.10,  # recovery probability per step
    max_steps: int = 50,
    rng_seed: int = SEED,
):
    """Run a single SIR simulation. Returns per-step (S, I, R) counts."""
    rng = random.Random(rng_seed)
    nodes = list(graph.nodes())
    state = {n: "S" for n in nodes}
    state[seed_node] = "I"

    history = []
    for _ in range(max_steps):
        s = sum(1 for v in state.values() if v == "S")
        i = sum(1 for v in state.values() if v == "I")
        r = sum(1 for v in state.values() if v == "R")
        history.append((s, i, r))
        if i == 0:
            break

        new_state = state.copy()
        infected_nodes = [n for n, st in state.items() if st == "I"]

        for inf_node in infected_nodes:
            # Try to infect successors
            for neighbor in graph.successors(inf_node):
                if state[neighbor] == "S":
                    edge_w = graph[inf_node][neighbor].get("weight", 1)
                    prob = min(beta * edge_w / 3, 0.95)  # scale by weight
                    if rng.random() < prob:
                        new_state[neighbor] = "I"
            # Possibly recover
            if rng.random() < gamma:
                new_state[inf_node] = "R"

        state = new_state

    # Pad to max_steps length
    last = history[-1]
    while len(history) < max_steps:
        history.append(last)

    return pd.DataFrame(history, columns=["S", "I", "R"])


# Select seed nodes
top_influencer_node = top_influencers.index[0]
median_score = centrality_df["influencer_score"].median()
random_node = (
    centrality_df
    .assign(diff=lambda d: (d["influencer_score"] - median_score).abs())
    .sort_values("diff")
    .index[0]
)

print(f"Top influencer seed: {top_influencer_node} (score={centrality_df.loc[top_influencer_node, 'influencer_score']:.4f})")
print(f"Random/median seed:  {random_node} (score={centrality_df.loc[random_node, 'influencer_score']:.4f})")

sir_top    = sir_simulation(DG, seed_node=top_influencer_node)
sir_random = sir_simulation(DG, seed_node=random_node)

print(f"\nTop influencer — peak infected: {sir_top['I'].max()} | total reached: {sir_top['R'].iloc[-1]}")
print(f"Random user    — peak infected: {sir_random['I'].max()} | total reached: {sir_random['R'].iloc[-1]}")

In [ ]:
# Plot SIR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
N = DG.number_of_nodes()

for ax, sir_df, label, color in [
    (axes[0], sir_top,    f"Top Influencer ({top_influencer_node})", "#e74c3c"),
    (axes[1], sir_random, f"Average User ({random_node})",           "#3498db"),
]:
    steps = range(len(sir_df))
    ax.fill_between(steps, sir_df["S"] / N, alpha=0.25, color="#2ecc71", label="Susceptible")
    ax.fill_between(steps, sir_df["I"] / N, alpha=0.60, color=color,    label="Infected")
    ax.fill_between(steps, sir_df["R"] / N, alpha=0.35, color="#95a5a6", label="Recovered")
    ax.plot(steps, sir_df["I"] / N, color=color, linewidth=2)
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_xlabel("Time Step")
    ax.set_ylabel("Fraction of Users")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)

fig.suptitle("SIR Information Spread Simulation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sir_simulation.png", bbox_inches="tight")
plt.show()
print("Figure saved: sir_simulation.png")

---
## 10. Network Visualizations

We draw the network using a **spring layout** for the full graph and a **community-colored** view, with node size proportional to the influencer score.

In [ ]:
# Precompute layout (spring layout on undirected graph)
pos = nx.spring_layout(G, weight="weight", seed=SEED, k=0.45)

In [ ]:
# ── Figure 1: Colored by Persona ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))

node_colors = [PERSONA_CONFIG[user_persona[n]]["color"] for n in G.nodes()]
node_sizes  = [300 * centrality_df.loc[n, "influencer_score"] + 30 for n in G.nodes()]

nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.85,
)
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color="#cccccc",
    alpha=0.4,
    width=0.6,
    arrows=False,
)

# Label only top-10 influencers
top10 = top_influencers.head(10).index.tolist()
labels = {n: n for n in top10}
nx.draw_networkx_labels(
    G, pos, labels=labels, ax=ax,
    font_size=7, font_color="black", font_weight="bold",
)

# Legend
legend_handles = [
    plt.scatter([], [], s=80, color=cfg["color"], label=p)
    for p, cfg in PERSONA_CONFIG.items()
]
ax.legend(handles=legend_handles, title="Persona", loc="lower left", fontsize=9)
ax.set_title("Social Network — Colored by User Persona\n(node size ∝ influencer score)", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("network_by_persona.png", bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved: network_by_persona.png")

In [ ]:
# ── Figure 2: Colored by Community ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))

# Assign a color per community
comm_ids = sorted(set(partition.values()))
cmap = plt.get_cmap("tab20", len(comm_ids))
comm_color_map = {c: mcolors.to_hex(cmap(i)) for i, c in enumerate(comm_ids)}
node_colors_comm = [comm_color_map[partition[n]] for n in G.nodes()]

nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors_comm,
    node_size=node_sizes,
    alpha=0.85,
)
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color="#cccccc",
    alpha=0.35,
    width=0.6,
    arrows=False,
)
nx.draw_networkx_labels(
    G, pos, labels=labels, ax=ax,
    font_size=7, font_color="black", font_weight="bold",
)

legend_handles = [
    plt.scatter([], [], s=80, color=comm_color_map[c], label=f"Community {c} ({community_counts[c]} users)")
    for c in comm_ids
]
ax.legend(handles=legend_handles, title="Community", loc="lower left", fontsize=8)
ax.set_title(
    f"Social Network — Colored by Community ({method_name}, Q={modularity:.3f})\n(node size ∝ influencer score)",
    fontsize=14,
    fontweight="bold",
)
ax.axis("off")
plt.tight_layout()
plt.savefig("network_by_community.png", bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved: network_by_community.png")

In [ ]:
# ── Figure 3: Centrality Heat-Map overlay ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))

scores = [centrality_df.loc[n, "influencer_score"] for n in G.nodes()]
sc = nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=scores,
    cmap=plt.get_cmap("plasma"),
    node_size=node_sizes,
    alpha=0.9,
)
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color="#dddddd",
    alpha=0.3,
    width=0.5,
    arrows=False,
)
nx.draw_networkx_labels(
    G, pos, labels=labels, ax=ax,
    font_size=7, font_color="white", font_weight="bold",
)
plt.colorbar(sc, ax=ax, label="Influencer Score")
ax.set_title("Influencer Score Heatmap on Network", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("network_influencer_heatmap.png", bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved: network_influencer_heatmap.png")

---
## 11. Social Media Strategy Discussion

### 11.1 Summary of Key Findings

The graph analysis of this simulated social network reveals several actionable patterns:

| Finding | Insight |
|---|---|
| **Power Users dominate in-degree centrality** | A small group (~7% of users) receives a disproportionate share of interactions. Identifying and partnering with this group would maximize organic reach. |
| **Bridge Nodes score high on betweenness** | Despite lower raw follower counts, Bridge Nodes link communities that would otherwise be disconnected. They are crucial for cross-community content diffusion. |
| **High modularity (Q > 0.3)** | The network divides into distinct communities with limited inter-community ties. Content that "goes viral" within one community may not naturally cross into others without deliberate seeding. |
| **Low clustering for Bridge Nodes** | Bridge Nodes are structurally embedded between groups (low clustering), making them ideal amplifiers for campaigns targeting multiple audience segments. |
| **SIR simulation confirms influencer advantage** | Content seeded from the top influencer reaches significantly more users and peaks faster than content seeded from an average user. |

---

### 11.2 Recommendations for Social Media Strategy

#### 1. Prioritize Influencer Partnerships for Launch Campaigns
The composite influencer score identifies users who combine **reach** (in-degree), **prestige** (eigenvector), **brokerage** (betweenness), and **speed** (closeness). Partnering with top-ranked users for new product launches or announcements ensures the fastest and widest initial spread. A co-creation model (e.g., exclusive previews, collaborative content) tends to outperform paid sponsorships in organic engagement.

#### 2. Leverage Bridge Nodes to Penetrate New Communities
Bridge Nodes occupy bottlenecks between communities. Activating them — through early access, ambassador programmes, or personalized outreach — can break down information silos and bring a message to new audience segments that would not otherwise encounter it through organic sharing alone.

#### 3. Design Community-Specific Content
Because the network has well-separated communities (high modularity), generic "one-size-fits-all" content performs sub-optimally. Each detected community likely represents a distinct interest group or demographic. Tailoring messaging, format, and tone to each community and seeding it with community-internal influencers will increase relevance and engagement.

#### 4. Re-engage the Lurker Segment Deliberately
Lurkers represent ~33% of the network but contribute few outgoing interactions. While they appear passive, they are still *consumers* of content. Strategies to convert Lurkers include:
  - **Low-barrier CTAs** (polls, reactions, one-tap replies)
  - **Exclusive community events** that reward participation
  - **Personalized notifications** triggered by activity from users they already follow

#### 5. Monitor Network Evolution Over Time
A one-time snapshot provides strategic direction, but social networks evolve rapidly. Repeating this analysis monthly enables the detection of:
  - Emerging influencers (rising centrality scores)
  - Community fragmentation or merging (modularity shifts)
  - Changes in information-spread dynamics after algorithm or product updates

#### 6. Optimize Content Timing Using Closeness Centrality
Users with high closeness centrality can reach the rest of the network in the fewest steps. Scheduling and targeting content toward these users during peak activity windows compounds organic reach through the network's natural amplification structure.

---

### 11.3 Limitations & Next Steps

- **Synthetic data**: Real-world interaction data may have heavier-tailed degree distributions (power-law), temporal dynamics, and platform-specific nuances (algorithm curation, shadowbanning, paid boosts).
- **Static snapshot**: This analysis treats the network as static. A temporal network analysis would capture how relationships and influence evolve.
- **SIR simplification**: Real information spread involves content quality, sentiment, timing, and platform mechanics. More sophisticated epidemic models (e.g., SEIR, independent cascade) may better capture these dynamics.
- **Community labeling**: Communities are detected algorithmically; qualitative validation (interviewing users, inspecting content themes) is needed to attach real-world meaning to each cluster.

> **Conclusion**: Graph-based analysis of social networks provides a powerful, quantitative lens for strategic decision-making. By understanding who the key connectors are, how information flows, and where communities cluster, social media managers can design targeted, efficient, and measurable campaigns that outperform intuition-driven approaches.